# 00 - Model Used and Classification Head Architecture

This notebook documents the model architecture used by the current HuBERT temporal-relevance pipeline. It focuses on the part that matters for LeGrad adaptation: how Transformer hidden states become an emotion-class score.

## What This Notebook Checks

- Which model source the current executable scripts use.
- Whether the sequence-classification head can use a learned weighted sum of Transformer hidden layers.
- Where `projector`, temporal pooling, and `classifier` appear in the forward path.
- Why this differs from the ViT/CLIP architecture used by the original LeGrad implementation.

In [1]:
from pathlib import Path
import inspect
import textwrap

import pandas as pd
from IPython.display import HTML, Markdown, display
from transformers import AutoConfig

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# The current explanation scripts call load_hubert_emotion_model(), whose default is this model.
ACTIVE_MODEL_SOURCE = "superb/hubert-base-superb-er"

# The project YAML is older and may still point to the original SpeechXAI wav2vec2 checkpoint.
yaml_model_source = None
try:
    import yaml

    config_path = PROJECT_ROOT / "configs" / "default.yaml"
    if config_path.exists():
        project_config = yaml.safe_load(config_path.read_text())
        yaml_model_source = project_config.get("model", {}).get("source")
except Exception as exc:
    yaml_model_source = f"Could not read YAML: {exc}"

display(Markdown(f"""
**Active model used by current explanation scripts:** `{ACTIVE_MODEL_SOURCE}`

**Model source still present in `configs/default.yaml`:** `{yaml_model_source}`

The rest of this notebook inspects the active HuBERT model used by the current transformer-relevance scripts.
"""))

C:\Users\mateu\repos\gradient_based_speach_xai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



**Active model used by current explanation scripts:** `superb/hubert-base-superb-er`

**Model source still present in `configs/default.yaml`:** `superb/wav2vec2-base-superb-er`

The rest of this notebook inspects the active HuBERT model used by the current transformer-relevance scripts.


## Load the Model Configuration

This loads only the configuration, not the full model weights. It is fast and enough to inspect the architecture fields that control the classification head.

In [2]:
config = AutoConfig.from_pretrained(
    ACTIVE_MODEL_SOURCE,
    trust_remote_code=True,
)

fields = [
    ("model_type", getattr(config, "model_type", None)),
    ("architectures", getattr(config, "architectures", None)),
    ("num_hidden_layers", getattr(config, "num_hidden_layers", None)),
    ("num_attention_heads", getattr(config, "num_attention_heads", None)),
    ("hidden_size", getattr(config, "hidden_size", None)),
    ("classifier_proj_size", getattr(config, "classifier_proj_size", None)),
    ("use_weighted_layer_sum", getattr(config, "use_weighted_layer_sum", None)),
    ("num_labels", getattr(config, "num_labels", None)),
    ("id2label", getattr(config, "id2label", None)),
]

display(pd.DataFrame(fields, columns=["config field", "value"]))
display(Markdown("**Legend.** `config field` is the HuBERT/SUPERB configuration attribute inspected from the active model config; `value` is the loaded value for that attribute."))

,config field,value
0,model_type,hubert
1,architectures,[HubertForSequenceClassification]
2,num_hidden_layers,12
3,num_attention_heads,12
4,hidden_size,768
5,classifier_proj_size,256
6,use_weighted_layer_sum,True
7,num_labels,4
8,id2label,"{0: 'neu', 1: 'hap', 2: 'ang', 3: 'sad'}"


**Legend.** `config field` is the HuBERT/SUPERB configuration attribute inspected from the active model config; `value` is the loaded value for that attribute.

## HuBERT Sequence Classification Path

The important part for LeGrad is not the convolutional feature encoder alone. The convolutional stack mainly affects how Transformer tokens map back to waveform time. The layer-wise score issue comes later, in the classification path from Transformer hidden states to emotion logits.

In [3]:
weighted = bool(getattr(config, "use_weighted_layer_sum", False))
num_layers = getattr(config, "num_hidden_layers", "L")
hidden_size = getattr(config, "hidden_size", "D")
proj_size = getattr(config, "classifier_proj_size", "P")
num_labels = getattr(config, "num_labels", "C")

aggregation_label = (
    "learned softmax(layer_weights) mixture of H_0 ... H_L"
    if weighted
    else "last hidden states H_L"
)

display(HTML(f"""
<style>
.arch-wrap {{ font-family: system-ui, -apple-system, Segoe UI, sans-serif; }}
.arch-title {{ font-weight: 700; margin: 0 0 10px; color: #172033; }}
.arch-flow {{ display: flex; flex-wrap: wrap; align-items: center; gap: 8px; }}
.arch-box {{ border: 1px solid #b9c4d6; border-radius: 8px; padding: 10px 12px; background: #f8fafc; min-width: 145px; }}
.arch-box strong {{ display: block; color: #0f172a; font-size: 13px; }}
.arch-box span {{ color: #475569; font-size: 12px; }}
.arch-arrow {{ color: #64748b; font-weight: 700; }}
.arch-note {{ margin-top: 12px; padding: 10px 12px; border-left: 4px solid #2563eb; background: #eff6ff; color: #1e3a8a; }}
</style>
<div class='arch-wrap'>
  <div class='arch-title'>Active HuBERT emotion-classification path</div>
  <div class='arch-flow'>
    <div class='arch-box'><strong>raw waveform</strong><span>16 kHz audio samples</span></div>
    <div class='arch-arrow'>-&gt;</div>
    <div class='arch-box'><strong>conv feature encoder</strong><span>downsamples waveform into frames</span></div>
    <div class='arch-arrow'>-&gt;</div>
    <div class='arch-box'><strong>HuBERT Transformer</strong><span>{num_layers} layers, hidden size {hidden_size}</span></div>
    <div class='arch-arrow'>-&gt;</div>
    <div class='arch-box'><strong>layer selection</strong><span>{aggregation_label}</span></div>
    <div class='arch-arrow'>-&gt;</div>
    <div class='arch-box'><strong>projector</strong><span>{hidden_size} -> {proj_size}</span></div>
    <div class='arch-arrow'>-&gt;</div>
    <div class='arch-box'><strong>temporal mean pooling</strong><span>tokens -> utterance vector</span></div>
    <div class='arch-arrow'>-&gt;</div>
    <div class='arch-box'><strong>classifier</strong><span>{proj_size} -> {num_labels} emotion logits</span></div>
  </div>
  <div class='arch-note'>
    If weighted layer aggregation is enabled, the head is trained to consume a learned mixture of hidden layers, not one arbitrary individual layer H_l.
  </div>
</div>
"""))

## Inspect the Actual Hugging Face Forward Logic

The next cell prints the relevant source excerpt from `HubertForSequenceClassification.forward`. This is the code path that creates the representation consumed by the classification head.

In [4]:
from transformers.models.hubert.modeling_hubert import HubertForSequenceClassification

source = inspect.getsource(HubertForSequenceClassification.forward).splitlines()
keywords = [
    "use_weighted_layer_sum",
    "layer_weights",
    "projector",
    "pooled_output",
    "classifier",
]
matching = [i for i, line in enumerate(source) if any(keyword in line for keyword in keywords)]
start = max(0, min(matching) - 8)
end = min(len(source), max(matching) + 8)

excerpt = "\n".join(f"{i + 1:03d}: {source[i]}" for i in range(start, end))
display(Markdown("```python\n" + excerpt + "\n```"))

```python
017:             into a tensor of type `torch.FloatTensor`. See [`HubertProcessor.__call__`] for details.
018:         labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
019:             Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
020:             config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
021:             `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
022:         """
023: 
024:         return_dict = return_dict if return_dict is not None else self.config.use_return_dict
025:         output_hidden_states = True if self.config.use_weighted_layer_sum else output_hidden_states
026: 
027:         outputs = self.hubert(
028:             input_values,
029:             attention_mask=attention_mask,
030:             output_attentions=output_attentions,
031:             output_hidden_states=output_hidden_states,
032:             return_dict=return_dict,
033:         )
034: 
035:         if self.config.use_weighted_layer_sum:
036:             hidden_states = outputs[_HIDDEN_STATES_START_POSITION]
037:             hidden_states = torch.stack(hidden_states, dim=1)
038:             norm_weights = nn.functional.softmax(self.layer_weights, dim=-1)
039:             hidden_states = (hidden_states * norm_weights.view(-1, 1, 1)).sum(dim=1)
040:         else:
041:             hidden_states = outputs[0]
042: 
043:         hidden_states = self.projector(hidden_states)
044:         if attention_mask is None:
045:             pooled_output = hidden_states.mean(dim=1)
046:         else:
047:             padding_mask = self._get_feature_vector_attention_mask(hidden_states.shape[1], attention_mask)
048:             expand_padding_mask = padding_mask.unsqueeze(-1).repeat(1, 1, hidden_states.shape[2])
049:             hidden_states[~expand_padding_mask] = 0.0
050:             pooled_output = hidden_states.sum(dim=1) / padding_mask.sum(dim=1).view(-1, 1)
051: 
052:         logits = self.classifier(pooled_output)
053: 
054:         loss = None
055:         if labels is not None:
056:             loss_fct = CrossEntropyLoss()
057:             loss = loss_fct(logits.view(-1, self.config.num_labels), labels.view(-1))
058: 
059:         if not return_dict:
```

## Layer Mixing Is Not the Projector

The learned layer mixture and the projector are two different operations. The mixture combines hidden states from different HuBERT depths. The projector is a linear layer applied after that mixture, frame by frame, to reduce the feature dimension before temporal pooling and classification.

In [5]:
component_table = pd.DataFrame(
    [
        {
            "component": "weighted layer mixture",
            "stored parameter": "layer_weights, shape [13]",
            "input -> output": "[B, 13, T, 768] -> [B, T, 768]",
            "what it does": "combines H_0 ... H_12 across the layer axis using softmax(layer_weights)",
        },
        {
            "component": "projector",
            "stored parameter": "projector.weight [256, 768] and projector.bias [256]",
            "input -> output": "[B, T, 768] -> [B, T, 256]",
            "what it does": "applies a learned linear projection to each temporal frame; it does not mix layers or pool time",
        },
        {
            "component": "temporal pooling",
            "stored parameter": "none",
            "input -> output": "[B, T, 256] -> [B, 256]",
            "what it does": "averages projected frame vectors over valid time frames to create one utterance vector",
        },
        {
            "component": "classifier",
            "stored parameter": "classifier.weight [4, 256] and classifier.bias [4]",
            "input -> output": "[B, 256] -> [B, 4]",
            "what it does": "maps the pooled utterance vector to the four emotion logits",
        },
    ]
)

display(component_table.style.set_properties(**{"text-align": "left"}).hide(axis="index"))
display(Markdown("**Legend.** `component` names the operation in the HuBERT classification head; `stored parameter` lists the trainable checkpoint tensor, if one exists; `input -> output` shows the tensor shape change; `what it does` explains whether the operation mixes layers, changes feature dimension, pools time, or produces logits."))

display(Markdown("**Projector clarification.** In `H_l -> projector -> temporal pooling -> classifier`, `projector` means the existing `nn.Linear(768, 256)` layer from the classifier head. It is not the layer-mixing step. The adaptation is that a single layer representation `H_l` is passed through this same head path, while the trained HuBERT path normally passes `H_mix` through it."))

component,stored parameter,input -> output,what it does
weighted layer mixture,"layer_weights, shape [13]","[B, 13, T, 768] -> [B, T, 768]",combines H_0 ... H_12 across the layer axis using softmax(layer_weights)
projector,"projector.weight [256, 768] and projector.bias [256]","[B, T, 768] -> [B, T, 256]",applies a learned linear projection to each temporal frame; it does not mix layers or pool time
temporal pooling,none,"[B, T, 256] -> [B, 256]",averages projected frame vectors over valid time frames to create one utterance vector
classifier,"classifier.weight [4, 256] and classifier.bias [4]","[B, 256] -> [B, 4]",maps the pooled utterance vector to the four emotion logits


**Legend.** `component` names the operation in the HuBERT classification head; `stored parameter` lists the trainable checkpoint tensor, if one exists; `input -> output` shows the tensor shape change; `what it does` explains whether the operation mixes layers, changes feature dimension, pools time, or produces logits.

**Projector clarification.** In `H_l -> projector -> temporal pooling -> classifier`, `projector` means the existing `nn.Linear(768, 256)` layer from the classifier head. It is not the layer-mixing step. The adaptation is that a single layer representation `H_l` is passed through this same head path, while the trained HuBERT path normally passes `H_mix` through it.

## Why This Differs from the ViT/CLIP Case Used by LeGrad

Original LeGrad was designed around Vision Transformers. For ViT/CLIP, the intermediate tokens from a block can be aggregated and passed through the final visual projection to define a layer-wise target score. HuBERT speech classification has a different head design, especially when the classifier expects a learned mixture of layer outputs.

In [6]:
comparison = pd.DataFrame(
    [
        {
            "aspect": "normal model score",
            "ViT / CLIP": "final visual tokens -> CLS/mean -> visual projection -> score",
            "HuBERT / SUPERB": "H_mix from weighted layer mixture -> projector -> temporal mean pooling -> classifier",
        },
        {
            "aspect": "LeGrad layer-wise score",
            "ViT / CLIP": "H_l -> token aggregation -> final visual projection -> target score_l",
            "HuBERT / SUPERB": "single H_l -> same projector -> temporal pooling -> classifier is an adaptation; the trained path first builds H_mix",
        },
        {
            "aspect": "main complication",
            "ViT / CLIP": "intermediate patch tokens remain in the visual-token stream used by the projection",
            "HuBERT / SUPERB": "the head may be trained on H_mix = sum_l softmax(alpha)_l H_l, so a single H_l can be out of distribution for the head",
        },
        {
            "aspect": "output geometry",
            "ViT / CLIP": "patch relevance -> 2D image heatmap",
            "HuBERT / SUPERB": "temporal-token relevance -> audio time intervals",
        },
    ]
)

display(comparison.style.set_properties(**{"text-align": "left"}).hide(axis="index"))
display(Markdown("**Legend.** `aspect` names the architectural question being compared; `ViT / CLIP` describes the original LeGrad vision setting; `HuBERT / SUPERB` describes the corresponding speech-classification setting used here."))

aspect,ViT / CLIP,HuBERT / SUPERB
normal model score,final visual tokens -> CLS/mean -> visual projection -> score,H_mix from weighted layer mixture -> projector -> temporal mean pooling -> classifier
LeGrad layer-wise score,H_l -> token aggregation -> final visual projection -> target score_l,single H_l -> same projector -> temporal pooling -> classifier is an adaptation; the trained path first builds H_mix
main complication,intermediate patch tokens remain in the visual-token stream used by the projection,"the head may be trained on H_mix = sum_l softmax(alpha)_l H_l, so a single H_l can be out of distribution for the head"
output geometry,patch relevance -> 2D image heatmap,temporal-token relevance -> audio time intervals


**Legend.** `aspect` names the architectural question being compared; `ViT / CLIP` describes the original LeGrad vision setting; `HuBERT / SUPERB` describes the corresponding speech-classification setting used here.

## Checkpoint Tensor Inspection

The cells above use the configuration and installed `transformers` source code. The next cell downloads or reuses the real checkpoint tensor file and inspects the learned layer-mixture weights plus the classifier-head parameters without constructing the full HuBERT module.

In [7]:
from huggingface_hub import hf_hub_download
import torch

checkpoint_path = hf_hub_download(ACTIVE_MODEL_SOURCE, filename="pytorch_model.bin")
state_dict = torch.load(checkpoint_path, map_location="cpu")

head_keys = [
    key
    for key in state_dict
    if any(part in key for part in ["layer_weights", "projector", "classifier"])
]
head_params = pd.DataFrame(
    [
        {"parameter": key, "shape": tuple(state_dict[key].shape)}
        for key in head_keys
    ]
)
display(head_params)
display(Markdown("**Legend.** `parameter` is the checkpoint tensor used by the weighted layer mixture or classification head; `shape` gives its tensor dimensions in the saved checkpoint."))

raw_layer_weights = state_dict["layer_weights"].float()
normalized_layer_weights = torch.softmax(raw_layer_weights, dim=-1)
layer_weight_table = pd.DataFrame(
    {
        "representation": [f"H_{idx}" for idx in range(len(raw_layer_weights))],
        "raw layer_weight": raw_layer_weights.numpy(),
        "forward softmax coefficient": normalized_layer_weights.numpy(),
    }
)
display(layer_weight_table.style.format({"raw layer_weight": "{:.6f}", "forward softmax coefficient": "{:.6f}"}))
display(Markdown("**Legend.** `representation` identifies each hidden-state tensor in the mixture; `raw layer_weight` is the learned scalar stored in the checkpoint; `forward softmax coefficient` is computed from the raw weights during the forward pass and is the value multiplied by each `H_l` to build `H_mix`. The softmax coefficients are not stored separately, so fine-tuning updates the raw weights and the coefficients are recomputed."))

print(f"Checkpoint tensor file: {checkpoint_path}")
print(f"Number of checkpoint tensors: {len(state_dict)}")

,parameter,shape
0,layer_weights,"(13,)"
1,projector.weight,"(256, 768)"
2,projector.bias,"(256,)"
3,classifier.weight,"(4, 256)"
4,classifier.bias,"(4,)"


**Legend.** `parameter` is the checkpoint tensor used by the weighted layer mixture or classification head; `shape` gives its tensor dimensions in the saved checkpoint.

,representation,raw layer_weight,forward softmax coefficient
0,H_0,-0.061527,0.072963
1,H_1,-0.134860,0.067804
2,H_2,-0.014761,0.076456
3,H_3,-0.056741,0.073313
4,H_4,-0.077151,0.071832
5,H_5,-0.067647,0.072518
6,H_6,-0.029904,0.075307
7,H_7,-0.042740,0.074347
8,H_8,0.033603,0.080245
9,H_9,0.065392,0.082837


**Legend.** `representation` identifies each hidden-state tensor in the mixture; `raw layer_weight` is the learned scalar stored in the checkpoint; `forward softmax coefficient` is computed from the raw weights during the forward pass and is the value multiplied by each `H_l` to build `H_mix`. The softmax coefficients are not stored separately, so fine-tuning updates the raw weights and the coefficients are recomputed.

Checkpoint tensor file: C:\Users\mateu\.cache\huggingface\hub\models--superb--hubert-base-superb-er\snapshots\bac0e14e92f7f9fd56671c5060e572e883cad667\pytorch_model.bin
Number of checkpoint tensors: 216


## Takeaway

For ViT/CLIP, the original LeGrad layer-wise score can reuse the visual projection on intermediate block outputs. For HuBERT/SUPERB, the classifier head consumes a learned hidden-layer mixture before projection and pooling. The projector is the linear `768 -> 256` layer after that mixture, not the mixture itself. Therefore, applying `projector -> pooling -> classifier` to one individual layer `H_l` is a methodological adaptation that should be stated explicitly when implementing a temporal LeGrad variant.